# train1: temporary training notebook (main line at 21,600 steps on mix v3 = 200K mix + 200K FineWeb QA texts)

The advisor's synthetic FineWeb QA data used as **distillation texts** (distillation needs no labels, every text with teacher vectors is a training example). Mix v3 (`turkish_distill.fineweb_mix`, built locally, on Drive under `bitirme/data/mix/`): the 200K mix (v2) plus 200,000 FineWeb texts, 400,000 in total:
- 60K paragraph chunks (150-600 characters, median 286) and 30K long chunks (800-1500, median 950) of the web documents;
- 30K search queries (median 61 characters) and 30K questions (60): the 8K teacher encodes these with its query instruction (80,000 questions with the 20,000 of the 200K mix);
- 25K answers (87) and 25K summaries (141).
At most one text per kind and document, train documents only, decontaminated by `prepare_synth_qa` (13-gram, TR-MTEB + Mizan + two RAG sets). The short texts fill a gap of the mix: most benchmark texts are short.

Steps (each resumable):
1. **Encoding.** Plain BGE-M3 and plain 8K-embed on the 400K corpus. The first 19 shards (190,000 texts) are copied from the 200K caches (`--seed-from-corpus`: same texts at the same positions), so 21 shards per teacher are encoded. Then the instructed 8K teacher: only the 80,000 questions go through the model, the rest is copied from the plain 8K cache.
2. **Training** `main_v3_s21600`: the main-line recipe (1k-base student, BGE-M3 + instructed 8K-embed, lr 1e-4, batch 128, seed 42, 21,600 steps), `configs/mix_v3_two_instr8k.yaml`. The validation split (last 2,000 texts) is the one of the 200K runs.

Reference: `main_s21600_avg` (same recipe on the 200K mix), TR-MTEB **67.93**; the only change is the corpus. With 400K texts the run sees every text ~7 times instead of ~14. Only differences above 0.3 are read. If it gains, it does not yet tell "more data" from "FineWeb content" (a control with 200K other texts would). Re-running the cells after a disconnect resumes the step that was cut, then the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# 1) Teacher encoding for mix v3 (skips finished shards; the 19 prefix shards are copied, not encoded)
MIX = f'{DRIVE_ROOT}/data/mix'
V3, V2 = f'{MIX}/mix_v3_400k.jsonl', f'{MIX}/mix_v2_200k.jsonl'
assert os.path.exists(V3) and os.path.exists(f'{MIX}/mix_v3_400k_questions.jsonl'), 'mix v3 is not on Drive yet'
get_ipython().system(f'python -m turkish_distill.encode_teachers --config configs/stage1_two_bge_m3_modernbert_tr_embed.yaml '
                     f'--override data.corpus_path={V3} --seed-from-corpus {V2}')
get_ipython().system('python -m turkish_distill.encode_teachers --config configs/mix_v3_two_instr8k.yaml --only modernbert_tr_embed_instr')

# 2) Runs of this notebook. (name, config, extra overrides). A later override wins.
BASE = []
RUNS = [
    ('main_v3_s21600', 'configs/mix_v3_two_instr8k.yaml', []),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()